# 00 — Pilot Study: Reliability and Item Diagnostics (N ≈ 60)

**Input:** the raw pilot-study CSV — the same 33-column questionnaire used for the
main study, administered to a smaller pilot sample before full-scale data collection.

**What this notebook reports, and why:** at a pilot sample size (~60), a confirmatory
factor analysis of a 5-factor, 25-item model is not a defensible analysis — the
indicator-to-parameter ratio would be far too thin to trust. The standard,
correct division of labor is: **the pilot establishes reliability and item quality;
factorial and structural validity are assessed in the full-scale study.** This notebook
therefore reports:

1. Sample description
2. Reliability per construct (Cronbach's α, McDonald's ω) — with a bootstrap CI on α
3. Corrected item-total correlations per construct
4. Alpha-if-item-deleted
5. Item-level descriptives (mean, SD, skew) to check for floor/ceiling effects

**Before running:** set `PILOT_CSV_PATH` below to your pilot data file.

In [ ]:
!pip install semopy

In [ ]:
import pandas as pd
import numpy as np

PILOT_CSV_PATH = "AI_Literacy_CT_First_60_Samples.csv"  # <-- CHANGE THIS to your pilot file's path/name

df = pd.read_csv(PILOT_CSV_PATH)
print("Pilot raw N =", len(df))
print("Columns:", df.shape[1])
if df.shape[1] != 33:
    print("WARNING: expected 33 columns (same questionnaire as the main study) — "
          "got", df.shape[1], ". Check this is the correct, unmodified export.")

## Column map (identical to the main study — see `01_data_cleaning.ipynb`)

- 0: Timestamp · 1: information-sheet confirmation · 2: consent · 3: level of study ·
  4: field of study · 5: daily AI usage time
- 6–10: Metacognitive Awareness (MMR), item 10 reverse-worded
- 11: attention check 1 ("Disagree")
- 12–16: AI Usage (AIU), item 12 reverse-worded
- 17–21: AI Literacy (AIL), item 21 reverse-worded
- 22–26: Cognitive Offloading (CO), item 26 reverse-worded
- 27: attention check 2 ("Almost Always")
- 28–32: Critical Thinking Engagement (CTE), item 32 reverse-worded

## Step 1 — Apply the same consent, attention-check, and straightlining screen as the main study

In [ ]:
df1 = df[df.iloc[:, 2] == "Yes, I consent to participate."].copy()
print("After consent filter:", len(df1), " (dropped", len(df) - len(df1), ")")

ac1_pass = df1.iloc[:, 11] == "Disagree"
ac2_pass = df1.iloc[:, 27] == "Almost Always"
print("Failed attention check 1:", (~ac1_pass).sum())
print("Failed attention check 2:", (~ac2_pass).sum())
df2 = df1[ac1_pass & ac2_pass].copy()
print("After attention-check filter:", len(df2), " (dropped", len(df1) - len(df2), ")")

In [ ]:
AGREE_MAP = {"Strongly Disagree": 1, "Disagree": 2, "Neutral": 3, "Agree": 4, "Strongly Agree": 5}
FREQ_MAP  = {"Never": 1, "Rarely": 2, "Sometimes": 3, "Often": 4, "Almost Always": 5}

mmr_cols = list(range(6, 11))
aiu_cols = list(range(12, 17))
ail_cols = list(range(17, 22))
co_cols  = list(range(22, 27))
cte_cols = list(range(28, 33))
all_cols = mmr_cols + aiu_cols + ail_cols + co_cols + cte_cols
agree_idx = set(mmr_cols) | set(ail_cols)

numeric = pd.DataFrame(index=df2.index)
for i in all_cols:
    col = df2.iloc[:, i]
    mapping = AGREE_MAP if i in agree_idx else FREQ_MAP
    numeric[f"c{i}"] = col.map(mapping)

print("Recoded 25 items for", len(numeric), "pilot respondents.")
print("Any unmapped (NaN) values:", numeric.isna().sum().sum())

## Step 2 — Straightlining check

In [ ]:
row_variance = numeric.var(axis=1)
straightliners = row_variance < 0.01
print("Straightliners detected:", straightliners.sum())
numeric = numeric[~straightliners].copy()
print("Final pilot analytic N =", len(numeric))

## Step 3 — Sample description

In [ ]:
meta_pilot = df2.loc[numeric.index]
print("Level of study:\n", meta_pilot.iloc[:, 3].value_counts(), "\n")
print("Field of study:\n", meta_pilot.iloc[:, 4].value_counts(), "\n")
print("Daily AI usage:\n", meta_pilot.iloc[:, 5].value_counts())

## Step 4 — Reliability per construct (Cronbach's α, McDonald's ω)

A bootstrap 95% CI is reported alongside α, since reliability point estimates are
noticeably less stable at N≈60 than at the main study's N=339.

In [ ]:
from semopy import Model

constructs_5item = {
    "MMR": ["c6", "c7", "c8", "c9", "c10"],
    "AIU": ["c12", "c13", "c14", "c15", "c16"],
    "AIL": ["c17", "c18", "c19", "c20", "c21"],
    "CO":  ["c22", "c23", "c24", "c25", "c26"],
    "CTE": ["c28", "c29", "c30", "c31", "c32"],
}
reverse_items = {"MMR": "c10", "AIU": "c12", "AIL": "c21", "CO": "c26", "CTE": "c32"}

def cronbach_alpha(items_df):
    k = items_df.shape[1]
    item_vars = items_df.var(axis=0, ddof=1)
    total_var = items_df.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - item_vars.sum() / total_var)

def mcdonald_omega(items_df, construct_name):
    items = list(items_df.columns)
    desc = f"{construct_name} =~ " + " + ".join(items)
    m = Model(desc)
    m.fit(items_df)
    ins = m.inspect(std_est=True)
    ins["Est. Std"] = pd.to_numeric(ins["Est. Std"], errors="coerce")
    loadings = ins[(ins["op"] == "~") & (ins["rval"] == construct_name)]["Est. Std"].values
    sum_load = loadings.sum()
    sum_err = (1 - loadings**2).sum()
    return (sum_load**2) / (sum_load**2 + sum_err)

for name, items in constructs_5item.items():
    tmp = numeric[items].copy()
    tmp[reverse_items[name]] = 6 - tmp[reverse_items[name]]  # reverse-score before computing reliability
    a = cronbach_alpha(tmp)
    o = mcdonald_omega(tmp, name)
    print(f"{name}: alpha={a:.3f}  omega={o:.3f}  (5-item, reverse-scored)")

## Bootstrap 95% CI for alpha (2,000 resamples)

In [ ]:
np.random.seed(42)
n_boot = 2000
n_pilot = len(numeric)

for name, items in constructs_5item.items():
    tmp = numeric[items].copy()
    tmp[reverse_items[name]] = 6 - tmp[reverse_items[name]]
    boot_alphas = []
    for _ in range(n_boot):
        idx = np.random.choice(n_pilot, size=n_pilot, replace=True)
        boot_alphas.append(cronbach_alpha(tmp.iloc[idx]))
    lo, hi = np.percentile(boot_alphas, [2.5, 97.5])
    print(f"{name}: alpha 95% CI = [{lo:.3f}, {hi:.3f}]")

## Step 5 — Item-total correlations and alpha-if-item-deleted

Items are reverse-scored **before** this diagnostic, consistent with the
approach adopted throughout this study.

In [ ]:
def item_total_correlations(items_df):
    results = {}
    for col in items_df.columns:
        rest = items_df.drop(columns=[col]).sum(axis=1)
        results[col] = items_df[col].corr(rest)
    return results

def alpha_if_deleted(items_df):
    results = {}
    for col in items_df.columns:
        results[col] = cronbach_alpha(items_df.drop(columns=[col]))
    return results

for name, items in constructs_5item.items():
    tmp = numeric[items].copy()
    tmp[reverse_items[name]] = 6 - tmp[reverse_items[name]]
    print(f"\n{name} (reverse-scored):")
    itc = item_total_correlations(tmp)
    aid = alpha_if_deleted(tmp)
    full_alpha = cronbach_alpha(tmp)
    weakest = min(itc, key=itc.get)
    for item in items:
        marker = ("  <-- reverse item" if item == reverse_items[name] else "") + ("  [WEAKEST]" if item == weakest else "")
        print(f"  {item}: item-total r={itc[item]:+.3f}   alpha-if-deleted={aid[item]:.3f}{marker}")
    print(f"  (full-scale alpha, reverse-scored: {full_alpha:.3f})")

## Step 6 — Item-level descriptives (check for floor/ceiling effects)

In [ ]:
from scipy.stats import skew

desc_rows = []
for name, items in constructs_5item.items():
    for item in items:
        vals = numeric[item]
        desc_rows.append({
            "construct": name, "item": item,
            "mean": vals.mean(), "sd": vals.std(),
            "min": vals.min(), "max": vals.max(),
            "skew": skew(vals),
        })
desc_df = pd.DataFrame(desc_rows)
print(desc_df.round(2).to_string(index=False))

**Next notebook:** `01_data_cleaning.ipynb`